In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [26]:
import shutil, os

output_dir = "/kaggle/working/Features/Scanorama"

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    os.makedirs(output_dir)
    print(f"Cleared: {output_dir}")
else:
    print("Directory didn't exist, nothing to clear")

Cleared: /kaggle/working/Features/Scanorama


In [1]:
# ============================================================================
# SCANORAMA LATENT EMBEDDING EXTRACTION
# Input : per-sample raw-count CSVs  (from scVI_counts dataset)
#         spot_spatial_coordinates.csv
#         PNG patch directories
# Output: per-patch .pt tensors  shape (50,)  dtype float32
#         → mirrors HarmonyPCA and scVI embedding pipelines exactly
# ============================================================================

import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip("scanorama", "anndata", "scanpy")
# torch is pre-installed on Kaggle GPU kernels

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 84.3 MB/s eta 0:00:00


In [28]:
import os, glob, zipfile, warnings
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scanorama
import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────
CONFIG = dict(
    # Same CSVs exported by the R count-export notebook
    csv_root    = "/kaggle/input/notebooks/wanianaeem/scvi-latent-embeddings/scVI_counts",

    # PNG patches
    png_root    = "/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/.png patches/.png patches",

    # Barcode ↔ row/col lookup
    coords_csv  = "/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset/spot_spatial_coordinates.csv",

    # Where .pt files are saved
    output_root = "/kaggle/working/Features/Scanorama",

    samples = [
        "IU_PDA_HM11", "IU_PDA_HM13", "IU_PDA_T1",
        "IU_PDA_T11",  "IU_PDA_T3",   "IU_PDA_T4",
    ],

    # Scanorama hyperparameters
    dimred = 50,    # match HarmonyPCA (50) and scVI (50)
    hvgs   = 3000,  # highly variable genes used for integration
)

os.makedirs(CONFIG["output_root"], exist_ok=True)

print(f"CSV root    : {CONFIG['csv_root']}")
print(f"Output root : {CONFIG['output_root']}")
print(f"Samples     : {', '.join(CONFIG['samples'])}")
print(f"dimred      : {CONFIG['dimred']}")

CSV root    : /kaggle/input/notebooks/wanianaeem/scvi-latent-embeddings/scVI_counts
Output root : /kaggle/working/Features/Scanorama
Samples     : IU_PDA_HM11, IU_PDA_HM13, IU_PDA_T1, IU_PDA_T11, IU_PDA_T3, IU_PDA_T4
dimred      : 50


In [29]:
print("=" * 70)
print("LOADING RAW COUNTS → AnnData list")
print("=" * 70)

# Scanorama takes a LIST of AnnData objects (one per batch/sample)
# — do NOT concat before integration

adatas     = []   # one AnnData per sample, cells × genes
barcodes   = []   # list of barcode lists, parallel to adatas
sample_ids = []   # sample name per adata entry

for sample in CONFIG["samples"]:
    csv_path = os.path.join(CONFIG["csv_root"], f"{sample}.csv")

    if not os.path.exists(csv_path):
        print(f"  WARNING: not found — {csv_path}")
        continue

    print(f"  Loading {sample} ... ", end="", flush=True)

    # CSV written by R: rows=genes, cols=barcodes, first col is gene names (row index)
    df = pd.read_csv(csv_path, index_col=0)   # shape: genes × barcodes

    # AnnData expects cells × genes → transpose
    adata = ad.AnnData(X=df.values.T.astype(np.float32))
    adata.obs_names = df.columns.tolist()      # barcodes
    adata.var_names = df.index.tolist()        # genes
    adata.obs["sample"] = sample

    print(f"{adata.n_obs} spots × {adata.n_vars} genes")

    adatas.append(adata)
    barcodes.append(adata.obs_names.tolist())
    sample_ids.append(sample)

print(f"\nLoaded {len(adatas)} samples")

LOADING RAW COUNTS → AnnData list
  Loading IU_PDA_HM11 ... 3931 spots × 17893 genes
  Loading IU_PDA_HM13 ... 2182 spots × 17893 genes
  Loading IU_PDA_T1 ... 3530 spots × 17893 genes
  Loading IU_PDA_T11 ... 2777 spots × 17893 genes
  Loading IU_PDA_T3 ... 4354 spots × 17893 genes
  Loading IU_PDA_T4 ... 3621 spots × 17893 genes

Loaded 6 samples


In [30]:
print("=" * 70)
print("PREPROCESSING")
print("=" * 70)

# Normalize + log each sample independently
# Do NOT subset to HVGs — Scanorama handles gene selection internally
# (intersection-based HVG filtering across 6 samples leaves too few genes)

for adata in adatas:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

print(f"Preprocessed {len(adatas)} samples")
for adata, sid in zip(adatas, sample_ids):
    print(f"  {sid:<20} {adata.n_obs} spots × {adata.n_vars} genes")

PREPROCESSING
Preprocessed 6 samples
  IU_PDA_HM11          3931 spots × 17893 genes
  IU_PDA_HM13          2182 spots × 17893 genes
  IU_PDA_T1            3530 spots × 17893 genes
  IU_PDA_T11           2777 spots × 17893 genes
  IU_PDA_T3            4354 spots × 17893 genes
  IU_PDA_T4            3621 spots × 17893 genes


In [31]:
print("=" * 70)
print("RUNNING SCANORAMA INTEGRATION")
print("=" * 70)

print(f"Integrating {len(adatas)} samples with dimred={CONFIG['dimred']} ... ")

# Step 1: integrate() → returns batch-corrected matrices (aligned, same gene space)
corrected, genes = scanorama.integrate(
    [adata.X for adata in adatas],
    [adata.var_names.tolist() for adata in adatas],
    dimred = CONFIG["dimred"],
    hvg    = CONFIG["hvgs"],
)

# Step 2: dimensionality_reduce() → PCA-like low-dim embedding on corrected matrices
embeddings = scanorama.dimensionality_reduce(corrected, dimred=CONFIG["dimred"])

print("\nScanorama integration complete")
for emb, sid in zip(embeddings, sample_ids):
    print(f"  {sid:<20} embedding shape: {np.array(emb).shape}")

RUNNING SCANORAMA INTEGRATION
Integrating 6 samples with dimred=50 ... 
Found 17893 genes among all datasets
[[0.         0.27726856 0.11898017 0.07778178 0.02595315 0.00483338]
 [0.         0.         0.20439963 0.05912007 0.02703941 0.01283226]
 [0.         0.         0.         0.22889518 0.33031161 0.01331445]
 [0.         0.         0.         0.         0.10046813 0.2603529 ]
 [0.         0.         0.         0.         0.         0.04391052]
 [0.         0.         0.         0.         0.         0.        ]]
Processing datasets (2, 4)
Processing datasets (0, 1)
Processing datasets (3, 5)
Processing datasets (2, 3)
Processing datasets (1, 2)
Processing datasets (0, 2)
Processing datasets (3, 4)

Scanorama integration complete
  IU_PDA_HM11          embedding shape: (3931, 50)
  IU_PDA_HM13          embedding shape: (2182, 50)
  IU_PDA_T1            embedding shape: (3530, 50)
  IU_PDA_T11           embedding shape: (2777, 50)
  IU_PDA_T3            embedding shape: (4354, 50)


In [32]:
print("=" * 70)
print("BUILDING BARCODE → EMBEDDING LOOKUP")
print("=" * 70)

# embeddings is a list of arrays (one per sample)
# barcodes  is a parallel list of barcode lists
# Flatten both into a single dict: barcode → embedding vector

barcode_to_emb = {}

for emb_array, bc_list, sid in zip(embeddings, barcodes, sample_ids):
    emb_np = np.array(emb_array, dtype=np.float32)  # (n_spots, 50)

    for j, bc in enumerate(bc_list):
        barcode_to_emb[bc] = emb_np[j]

print(f"Total barcode→embedding entries : {len(barcode_to_emb)}")
print(f"Embedding dim                   : {next(iter(barcode_to_emb.values())).shape}")

BUILDING BARCODE → EMBEDDING LOOKUP
Total barcode→embedding entries : 20395
Embedding dim                   : (50,)


In [33]:
print("=" * 70)
print("BUILDING ROW/COL → BARCODE LOOKUP")
print("=" * 70)

coords = pd.read_csv(CONFIG["coords_csv"])

# rc_key = "<image>_<row>_<col>"  →  barcode
coords["rc_key"] = (
    coords["image"].astype(str) + "_" +
    coords["row"].astype(str)   + "_" +
    coords["col"].astype(str)
)
rc_to_barcode = dict(zip(coords["rc_key"], coords["spot_barcode"]))

print(f"Loaded {len(rc_to_barcode)} coordinate entries")


def parse_row_col(filename: str):
    """Extract row, col from  <SAMPLE>_patch-XXXXXX_<ROW>_<COL>.png"""
    stem  = os.path.splitext(os.path.basename(filename))[0]
    parts = stem.split("_")
    return int(parts[-2]), int(parts[-1])

BUILDING ROW/COL → BARCODE LOOKUP
Loaded 91496 coordinate entries


In [34]:
print("=" * 70)
print("SAVING PER-PATCH .pt EMBEDDINGS")
print("=" * 70)

total_saved   = 0
total_missing = 0
total_failed  = 0

for sample in CONFIG["samples"]:

    print(f"\n{'=' * 70}")
    print(f"Processing: {sample}")
    print("=" * 70)

    png_dir = os.path.join(CONFIG["png_root"], sample)
    if not os.path.isdir(png_dir):
        print(f"  PNG directory not found: {png_dir}")
        continue

    patches = glob.glob(os.path.join(png_dir, "*.png"))
    print(f"  Found {len(patches)} patches")
    if not patches:
        continue

    out_dir = os.path.join(CONFIG["output_root"], sample)
    os.makedirs(out_dir, exist_ok=True)

    # Resume: skip already saved .pt files
    existing = {
        os.path.splitext(f)[0]
        for f in os.listdir(out_dir)
        if f.endswith(".pt")
    }
    to_process = [
        p for p in patches
        if os.path.splitext(os.path.basename(p))[0] not in existing
    ]

    if not to_process:
        print(f"  All {len(existing)} patches already processed")
        continue

    print(f"  Processing {len(to_process)} new patches (existing: {len(existing)})")

    saved = missing = failed = 0

    for patch_path in tqdm(to_process, desc=sample, leave=False):
        patch_name = os.path.splitext(os.path.basename(patch_path))[0]

        try:
            row, col = parse_row_col(patch_path)
        except Exception:
            failed += 1
            continue

        # row/col → barcode
        rc_key  = f"{sample}_{row}_{col}"
        barcode = rc_to_barcode.get(rc_key)
        if barcode is None:
            missing += 1
            continue

        # barcode → embedding
        emb_np = barcode_to_emb.get(barcode)
        if emb_np is None:
            missing += 1
            continue

        pt_path = os.path.join(out_dir, f"{patch_name}.pt")
        try:
            emb = torch.tensor(emb_np)   # shape (50,), float32
            torch.save(emb, pt_path)
            saved += 1
        except Exception as e:
            print(f"  Error on {patch_name}: {e}")
            failed += 1

    print(f"  Saved   : {saved} / {len(to_process)}")
    if missing: print(f"  Missing : {missing} (barcode not in lookup)")
    if failed:  print(f"  Failed  : {failed}")

    total_saved   += saved
    total_missing += missing
    total_failed  += failed

print(f"\n{'=' * 70}")
print(f"TOTAL SAVED   : {total_saved}")
print(f"TOTAL MISSING : {total_missing}")
print(f"TOTAL FAILED  : {total_failed}")
print(f"Output        : {CONFIG['output_root']}")

SAVING PER-PATCH .pt EMBEDDINGS

Processing: IU_PDA_HM11
  Found 3931 patches
  Processing 3931 new patches (existing: 0)


IU_PDA_HM11:   0%|          | 0/3931 [00:00<?, ?it/s]

  Saved   : 3931 / 3931

Processing: IU_PDA_HM13
  Found 2182 patches
  Processing 2182 new patches (existing: 0)


IU_PDA_HM13:   0%|          | 0/2182 [00:00<?, ?it/s]

  Saved   : 2182 / 2182

Processing: IU_PDA_T1
  Found 3530 patches
  Processing 3530 new patches (existing: 0)


...

  Saved   : 2777 / 2777

Processing: IU_PDA_T3
  Found 4354 patches
  Processing 4354 new patches (existing: 0)


IU_PDA_T3:   0%|          | 0/4354 [00:00<?, ?it/s]

  Saved   : 4354 / 4354

Processing: IU_PDA_T4
  Found 3621 patches
  Processing 3621 new patches (existing: 0)


IU_PDA_T4:   0%|          | 0/3621 [00:00<?, ?it/s]

  Saved   : 3621 / 3621

TOTAL SAVED   : 20395
TOTAL MISSING : 0
TOTAL FAILED  : 0
Output        : /kaggle/working/Features/Scanorama


In [35]:
print("=" * 70)
print("ZIPPING .pt FILES")
print("=" * 70)

zip_path = "/kaggle/working/scanorama_latent_pt_embeddings.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for sample in CONFIG["samples"]:
        pt_files = glob.glob(os.path.join(CONFIG["output_root"], sample, "*.pt"))
        for fpath in tqdm(pt_files, desc=sample, leave=False):
            arcname = os.path.relpath(fpath, "/kaggle/working")
            zf.write(fpath, arcname)

size_mb = os.path.getsize(zip_path) / 1e6
print(f"Done — {zip_path}")
print(f"Size : {size_mb:.1f} MB")

ZIPPING .pt FILES


IU_PDA_HM11:   0%|          | 0/3931 [00:00<?, ?it/s]

IU_PDA_HM13:   0%|          | 0/2182 [00:00<?, ?it/s]

IU_PDA_T1:   0%|          | 0/3530 [00:00<?, ?it/s]

IU_PDA_T11:   0%|          | 0/2777 [00:00<?, ?it/s]

IU_PDA_T3:   0%|          | 0/4354 [00:00<?, ?it/s]

IU_PDA_T4:   0%|          | 0/3621 [00:00<?, ?it/s]

Done — /kaggle/working/scanorama_latent_pt_embeddings.zip
Size : 20.6 MB


In [36]:
print("=" * 70)
print("VERIFICATION")
print("=" * 70)

# Per-sample counts
print("\nPer-sample .pt counts:")
for sample in CONFIG["samples"]:
    count = len(glob.glob(os.path.join(CONFIG["output_root"], sample, "*.pt")))
    print(f"  • {sample:<20} {count} embeddings")

# Spot-check one file
sample_check = CONFIG["samples"][0]
pt_files = glob.glob(os.path.join(CONFIG["output_root"], sample_check, "*.pt"))

if pt_files:
    emb = torch.load(pt_files[0], map_location="cpu")
    print(f"\nSpot-check — {os.path.basename(pt_files[0])}")
    print(f"  Shape  : {tuple(emb.shape)}")
    print(f"  Dtype  : {emb.dtype}")
    print(f"  Min    : {emb.min().item():.6f}")
    print(f"  Max    : {emb.max().item():.6f}")
    print(f"  Mean   : {emb.mean().item():.6f}")
    print(f"  Std    : {emb.std().item():.6f}")
    print(f"  L2norm : {emb.norm().item():.6f}")
    print(f"  First 5 values: {emb[:5].tolist()}")
else:
    print("No .pt files found — check pipeline ran successfully")

VERIFICATION

Per-sample .pt counts:
  • IU_PDA_HM11          3931 embeddings
  • IU_PDA_HM13          2182 embeddings
  • IU_PDA_T1            3530 embeddings
  • IU_PDA_T11           2777 embeddings
  • IU_PDA_T3            4354 embeddings
  • IU_PDA_T4            3621 embeddings

Spot-check — IU_PDA_HM11_patch-000932_56_68.pt
  Shape  : (50,)
  Dtype  : torch.float32
  Min    : -0.083763
  Max    : 0.073949
  Mean   : -0.003905
  Std    : 0.030471
  L2norm : 0.215074
  First 5 values: [-0.001599730458110571, -0.08376313745975494, 0.07394901663064957, -0.04413837194442749, -0.023101678118109703]
